# Diffusion-LM Watermarking

In [1]:
import os
import sys
from pathlib import Path

def find_repo_root(start: Path) -> Path:
    for parent in [start] + list(start.parents):
        if (parent / 'requirements.txt').exists() and (parent / 'models').exists():
            return parent
    raise FileNotFoundError('Could not locate repo root (requirements.txt). Open the notebook from inside the repo.')

REPO_ROOT = find_repo_root(Path.cwd().resolve())
os.chdir(REPO_ROOT)
print('python:', sys.executable)
print('cwd (repo root):', Path.cwd())
print('has requirements.txt:', (REPO_ROOT / 'requirements.txt').exists())

python: c:\Program Files\Python313\python.exe
cwd (repo root): C:\Users\Santosh Kumar Singh\Desktop\diffusion-lm-watermarking
has requirements.txt: True


In [2]:
# Install dependencies into *this kernel's* environment
# If you already installed them, this is safe to re-run.
import subprocess

requirements_path = str((REPO_ROOT / 'requirements.txt').resolve())
print('Installing from:', requirements_path)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', requirements_path])

Installing from: C:\Users\Santosh Kumar Singh\Desktop\diffusion-lm-watermarking\requirements.txt


0

In [3]:
import torch

print('torch:', torch.__version__)
print('torch.version.cuda:', torch.version.cuda)
print('cuda_available:', torch.cuda.is_available())
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device selected:', device)
if device == 'cuda':
    print('gpu:', torch.cuda.get_device_name(0))
else:
    print('Running on CPU. If you expected GPU: CUDA requires an NVIDIA GPU + driver + CUDA-enabled PyTorch.')

torch: 2.7.1+cpu
torch.version.cuda: None
cuda_available: False
device selected: cpu
Running on CPU. If you expected GPU: CUDA requires an NVIDIA GPU + driver + CUDA-enabled PyTorch.


In [4]:
# Show CLI help (sanity check)
import sys
import subprocess

subprocess.check_call([sys.executable, 'models/diffusion_lm.py', '--help'])

0

## Smoke-test training (fast)

This runs a short training loop to verify the pipeline end-to-end. It writes a checkpoint under `checkpoints/`.

In [5]:
import sys
import subprocess

cmd = [
    sys.executable, 'models/diffusion_lm.py', 'train',
    '--device', device,
    '--epochs', '1',
    '--batch_size', '16',
    '--max_length', '64',
    '--steps', '25',
    '--max_train_batches', '200',
    '--log_every', '5',
]
print('Running:', ' '.join(cmd))

# Stream child process output into the notebook (more reliable than capture_output in some setups).
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
assert proc.stdout is not None
for line in proc.stdout:
    print(line, end='')
rc = proc.wait()
if rc != 0:
    raise RuntimeError(f'Training failed with exit code {rc}')

Running: c:\Program Files\Python313\python.exe models/diffusion_lm.py train --device cpu --epochs 1 --batch_size 16 --max_length 64 --steps 25 --max_train_batches 200 --log_every 5
Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
epoch=1 step=5/1485 loss=8.1208
epoch=1 step=10/1485 loss=7.5858
epoch=1 step=15/1485 loss=7.4709
epoch=1 step=20/1485 loss=7.3883
e

In [6]:
from pathlib import Path

ckpt_root = Path('checkpoints/bert-mlm-diffusion-baseline')
if not ckpt_root.exists():
    print('No checkpoints found yet at:', ckpt_root)
else:
    def epoch_num(p: Path) -> int:
        # Sort like epoch-2 < epoch-10 (numeric), fall back to name sort.
        try:
            return int(p.name.split('-')[-1])
        except Exception:
            return -1

    epochs = sorted([p for p in ckpt_root.iterdir() if p.is_dir()], key=lambda p: (epoch_num(p), p.name))
    print('Checkpoint folders:')
    for p in epochs:
        print(' -', p)
    if epochs:
        CKPT_DIR = epochs[-1]
        print('Latest checkpoint:', CKPT_DIR)

Checkpoint folders:
 - checkpoints\bert-mlm-diffusion-baseline\epoch-1
Latest checkpoint: checkpoints\bert-mlm-diffusion-baseline\epoch-1


## Sampling

After training, pick the checkpoint folder printed by training (example: `checkpoints/bert-mlm-diffusion-baseline/epoch-1`).

Update `CKPT_DIR` below and run.

In [7]:
from pathlib import Path

# Uses CKPT_DIR from the previous cell if available; otherwise fall back to epoch-1.
CKPT_DIR = globals().get('CKPT_DIR', Path('checkpoints/bert-mlm-diffusion-baseline/epoch-1'))
print('exists:', Path(CKPT_DIR).exists(), 'path:', CKPT_DIR)

exists: True path: checkpoints\bert-mlm-diffusion-baseline\epoch-1


In [8]:
import sys
import subprocess
from pathlib import Path

ckpt = str(Path(CKPT_DIR))
cmd = [
    sys.executable, 'models/diffusion_lm.py', 'sample',
    '--ckpt', ckpt,
    '--device', device,
    '--length', '48',
    '--steps', '25',
    '--temperature', '1.0',
    '--top_k', '50',
]
print('Running:', ' '.join(cmd))

# Stream child process output into the notebook so you see the samples.
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
assert proc.stdout is not None
for line in proc.stdout:
    print(line, end='')
rc = proc.wait()
if rc != 0:
    raise RuntimeError(f'Sampling failed with exit code {rc}')

Running: c:\Program Files\Python313\python.exe models/diffusion_lm.py sample --ckpt checkpoints\bert-mlm-diffusion-baseline\epoch-1 --device cpu --length 48 --steps 25 --temperature 1.0 --top_k 50
of the a was,, the, ) ", of @ at with @, was from to, the @ for the on the the. from in ' @, to on a was by in of which of the in,


## Inspect forward masking

Shows an original vs masked example from WikiText-2 at a chosen diffusion timestep `t`.

In [9]:
import sys
import subprocess

cmd = [
    sys.executable, 'models/diffusion_lm.py', 'inspect-mask',
    '--split', 'train',
    '--idx', '0',
    '--max_length', '24',
    '--steps', '5',
    '--min_mask_prob', '0.2',
    '--max_mask_prob', '0.8',
    '--t', '4',
    '--device', 'cpu',
]
print('Running:', ' '.join(cmd))

# Stream child process output into the notebook.
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
assert proc.stdout is not None
for line in proc.stdout:
    print(line, end='')
rc = proc.wait()
if rc != 0:
    raise RuntimeError(f'Inspect-mask failed with exit code {rc}')

Running: c:\Program Files\Python313\python.exe models/diffusion_lm.py inspect-mask --split train --idx 0 --max_length 24 --steps 5 --min_mask_prob 0.2 --max_mask_prob 0.8 --t 4 --device cpu
split=train idx=0 max_length=24
t=4/5  p_t=0.650  masked=6/7

ORIGINAL (decoded):
= valkyria chronicles iii =

MASKED (decoded):
chronicles

TOKENS (original):
['[CLS]', '=', 'val', '##ky', '##ria', 'chronicles', 'iii', '=', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']

TOKENS (masked):
['[CLS]', '[MASK]', '[MASK]', '[MASK]', '[MASK]', 'chronicles', '[MASK]', '[MASK]', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']


## Step-by-step reconstruction (denoising) demo

This section shows two ways to reconstruct after training:

1. **Single-step denoising**: give the model a masked input $x_t$ and timestep `t`, and take its prediction as $\hat{x}_0$ in one shot.
2. **Iterative denoising (D3PM-style)**: starting from $x_t$, repeatedly unmask a subset of the remaining `[MASK]` tokens while stepping `t → t-1`, printing the intermediate text each step.

In [10]:
import torch
from pathlib import Path
from transformers import BertTokenizerFast
from models.diffusion_lm import (
    TimestepConditionedBertForMaskedLM,
    mask_prob_for_t,
    make_noised_example,
    resolve_device,
    set_seed,
    TrainConfig,
    )

# ---- Settings ----
CKPT_DIR = globals().get('CKPT_DIR', Path('checkpoints/bert-mlm-diffusion-baseline/epoch-1'))
ckpt = str(Path(CKPT_DIR))
device = resolve_device(globals().get('device', 'cpu'))
steps = 25
min_mask_prob = 0.15
max_mask_prob = 0.95

# Choose ONE of these inputs:
clean_text = "The quick brown fox jumps over the lazy dog in the park."
masked_text = None  # e.g. "The quick [MASK] fox jumps over the [MASK] dog."

# Reconstruction behavior
t_start = 20        # starting diffusion timestep (1..steps)
temperature = 0.7   # lower = more conservative
top_k = 50

set_seed(1234)
print('ckpt:', ckpt)
print('device:', device)

tokenizer = BertTokenizerFast.from_pretrained(ckpt)
model = TimestepConditionedBertForMaskedLM.from_pretrained(ckpt, num_steps=steps, device=device)
model.eval()

def encode_text(text: str) -> torch.Tensor:
    enc = tokenizer(
        text,
        add_special_tokens=True,
        truncation=True,
        max_length=64,
        padding='max_length',
        return_tensors='pt',
    )
    return enc['input_ids'][0]  # [L]

def decode_ids(input_ids_1d: torch.Tensor) -> str:
    return " ".join(tokenizer.decode(input_ids_1d, skip_special_tokens=True).split())

# Build x0 and x_t
if masked_text is not None:
    # Replace user-friendly [MASK] with the tokenizer's mask token if needed.
    mt = masked_text.replace('[MASK]', tokenizer.mask_token)
    x0 = encode_text(clean_text).to(device)
    x_t = encode_text(mt).to(device)
    # Infer t_start only from user choice (we keep the provided t_start)
    # Note: x_t may not have the "right" number of masks for that t; that's okay for visualization.
    mask_positions = x_t.eq(tokenizer.mask_token_id)
else:
    x0 = encode_text(clean_text).to(device)
    x_t, mask_positions = make_noised_example(
        x0,
        tokenizer=tokenizer,
        t_int=t_start,
        steps=steps,
        min_mask_prob=min_mask_prob,
        max_mask_prob=max_mask_prob,
        device=device,
    )

print('\nX0 (original):')
print(decode_ids(x0))
print(f"\nXT (masked at t={t_start}/{steps}):")
print(decode_ids(x_t))
print('num_masks:', int(mask_positions.sum().item()))

# ---- (A) Single-step denoising: x_t, t -> x0_hat in one shot ----
@torch.no_grad()
def single_step_denoise(x_t_1d: torch.Tensor, t_int: int) -> torch.Tensor:
    x_in = x_t_1d.unsqueeze(0)  # [1, L]
    t = torch.tensor([t_int], device=device)
    out = model(input_ids=x_in, t=t)
    logits = out['logits'][0]  # [L, V]
    logits = logits / max(float(temperature), 1e-6)
    if top_k and top_k > 0:
        topk_vals, topk_idx = torch.topk(logits, k=min(int(top_k), logits.size(-1)), dim=-1)
        filtered = torch.full_like(logits, fill_value=-float('inf'))
        filtered.scatter_(-1, topk_idx, topk_vals)
        logits = filtered
    probs = torch.softmax(logits, dim=-1)
    sampled = torch.multinomial(probs, num_samples=1).squeeze(-1)  # [L]
    x_hat = x_t_1d.clone()
    x_hat[mask_positions] = sampled[mask_positions]
    return x_hat

x0_hat = single_step_denoise(x_t, t_start)
print('\nSingle-step denoise (x_t, t -> x0_hat):')
print(decode_ids(x0_hat))

# ---- (B) Iterative denoising: print intermediate x_{t-1} as we unmask ----
@torch.no_grad()
def iterative_denoise_verbose(x_init_1d: torch.Tensor, t_init: int) -> list[str]:
    x = x_init_1d.clone().unsqueeze(0)  # [1, L]
    cls_id, sep_id, pad_id = tokenizer.cls_token_id, tokenizer.sep_token_id, tokenizer.pad_token_id
    texts = []
    texts.append(f"t={t_init}: {decode_ids(x[0])}")
    for t_int in range(int(t_init), 0, -1):
        t_tensor = torch.tensor([t_int], device=device)
        p_t = float(mask_prob_for_t(t_tensor, steps=steps, min_p=min_mask_prob, max_p=max_mask_prob)[0])
        p_prev = 0.0 if t_int == 1 else float(mask_prob_for_t(torch.tensor([t_int - 1], device=device), steps=steps, min_p=min_mask_prob, max_p=max_mask_prob)[0])
        p_t = max(p_t, 1e-6)
        keep_mask_prob = min(max(p_prev / p_t, 0.0), 1.0)

        special = x.eq(cls_id) | x.eq(sep_id)
        if pad_id is not None:
            special |= x.eq(pad_id)
        is_mask = x.eq(tokenizer.mask_token_id) & (~special)
        if not bool(is_mask.any()):
            texts.append(f"t={t_int-1}: {decode_ids(x[0])}")
            continue

        # Among currently-masked positions, decide which stay masked in the next step.
        stay_masked = (torch.rand_like(x.float()) < keep_mask_prob) & is_mask
        to_unmask = is_mask & (~stay_masked)
        if bool(to_unmask.any()):
            out = model(input_ids=x, t=t_tensor)
            logits = out['logits']  # [1, L, V]
            logits = logits / max(float(temperature), 1e-6)
            if top_k and top_k > 0:
                topk_vals, topk_idx = torch.topk(logits, k=min(int(top_k), logits.size(-1)), dim=-1)
                filtered = torch.full_like(logits, fill_value=-float('inf'))
                filtered.scatter_(-1, topk_idx, topk_vals)
                logits = filtered
            probs = torch.softmax(logits, dim=-1)
            sampled = torch.multinomial(probs.view(-1, probs.size(-1)), num_samples=1).view(1, -1)
            x[to_unmask] = sampled[to_unmask]
        # else: nothing to unmask this step
        texts.append(f"t={t_int-1}: {decode_ids(x[0])}")
    return texts

print('\nIterative denoise trace:')
trace = iterative_denoise_verbose(x_t, t_start)
for line in trace:
    print(line)

print("\nTip: if the trace changes too slowly, increase t_start or set max_mask_prob higher during training.")

C:\Users\Santosh Kumar Singh\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ckpt: checkpoints\bert-mlm-diffusion-baseline\epoch-1
device: cpu

X0 (original):
the quick brown fox jumps over the lazy dog in the park.

XT (masked at t=20/25):
fox dog the
num_masks: 10

Single-step denoise (x_t, t -> x0_hat):
= = = fox % = = = dog = the = =

Iterative denoise trace:
t=20: fox dog the
t=19: fox dog the
t=18: fox = dog the
t=17: fox = dog the
t=16: fox = dog the
t=15: fox = dog the
t=14: = fox = dog the
t=13: = fox = dog the
t=12: = fox = dog the
t=11: = fox = dog the
t=10: = fox = dog the
t=9: = fox = = dog the =
t=8: =mon fox = = dog the =
t=7: =mon fox = = dog the =
t=6: =mon fox = = dog the =
t=5: =mon fox = = dog the =
t=4: =mon fox = = dog the =
t=3: =mon fox = = = dog the =
t=2: =mon fox = = = dog the =
t=1: =mon fox = = = dog the =
t=0: = =mon fox = = = = dog = the = =

Tip: if the trace changes too slowly, increase t_start or set max_mask_prob higher during training.
